# Non-Housing Services Persistence with One Cyclical Control

This notebook estimates the same persistence object as the Fed note: the sum of
the coefficients on one- and two-quarter lagged inflation.

The baseline specification uses the output gap as the cyclical control. To test
whether a wage-based services control changes the result, change
`CYCLICAL_CONTROL` below from `"output_gap"` to `"eci_wage_growth"`.

When the wage control is selected, adjust `ECI_LAGS` to choose how many lagged
ECI wage-growth terms enter the regression. For example, use `[1]` for one lag
or `[1, 2, 3, 4]` for four lags. The model still includes only one cyclical
control family at a time: output gap or ECI wage growth.

In [ ]:
from pathlib import Path
import os
import pickle

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib

try:
    IN_IPYTHON = get_ipython() is not None
except NameError:
    IN_IPYTHON = False

if "__file__" in globals() and not IN_IPYTHON:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    display
except NameError:
    display = print


def find_root():
    candidates = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd().resolve())

    for path in candidates:
        if (path / "cache").exists() and (path / "Persistence").exists():
            return path
        if path.name.lower() == "persistence" and (path.parent / "cache").exists():
            return path.parent
    raise FileNotFoundError("Could not find workspace root with cache/ and Persistence/.")


ROOT = find_root()
CACHE = ROOT / "cache"
PERSISTENCE = ROOT / "Persistence"

BEA_TABLE = "T20304"
SERVICES_LINE = 28
ENERGY_LINE = 27
ECI_SERIES_ID = "CIS202S000000000I"
ECI_LAGS = [1]

# Baseline: "output_gap". Alternative: "eci_wage_growth".
# In the script, you can also set the CYCLICAL_CONTROL environment variable.
CYCLICAL_CONTROL = os.environ.get("CYCLICAL_CONTROL", "output_gap")

PRE_PANDEMIC_END = pd.Timestamp("2019-10-01")
POST_START = pd.Timestamp("2021-01-01")

CONTROL_OPTIONS = {
    "output_gap": {
        "terms": ["output_gap"],
        "label": "Output gap",
        "folder": "output_gap",
    },
    "eci_wage_growth": {
        "terms": [],
        "label": "",
        "folder": "",
    },
}

if os.environ.get("ECI_LAGS"):
    ECI_LAGS = [int(item.strip()) for item in os.environ["ECI_LAGS"].split(",") if item.strip()]

ECI_LAGS = sorted(dict.fromkeys(int(lag) for lag in ECI_LAGS))
if not ECI_LAGS or min(ECI_LAGS) < 1:
    raise ValueError("ECI_LAGS must contain positive integers, for example [1, 2].")

ECI_TERMS = [f"eci_wage_growth_l{lag}" for lag in ECI_LAGS]
ECI_LAG_TEXT = ", ".join(str(lag) for lag in ECI_LAGS)
CONTROL_OPTIONS["eci_wage_growth"] = {
    "terms": ECI_TERMS,
    "label": f"ECI services wage growth, lag(s) {ECI_LAG_TEXT}",
    "folder": "eci_wage_growth_l" + "_".join(str(lag) for lag in ECI_LAGS),
}

if CYCLICAL_CONTROL not in CONTROL_OPTIONS:
    raise ValueError(f"CYCLICAL_CONTROL must be one of {list(CONTROL_OPTIONS)}")

CONTROL = CONTROL_OPTIONS[CYCLICAL_CONTROL]
OUT = PERSISTENCE / "output" / "nonhousing_services_persistence_one_control" / CONTROL["folder"]
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)

RUN_STAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")


def save_csv(df, path, **kwargs):
    path = Path(path)
    try:
        df.to_csv(path, **kwargs)
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_{RUN_STAMP}{path.suffix}")
        df.to_csv(fallback, **kwargs)
        print(f"Could not overwrite {path.name}; wrote {fallback.name} instead.")
        return fallback

print(f"Using cyclical control: {CONTROL['label']}")
print(f"Outputs will be saved to: {OUT}")

## 1. Load Local Data

In [ ]:
def latest_bea_cache():
    files = sorted(CACHE.glob(f"bea_NIPA_{BEA_TABLE}_Q_*.csv"))
    if not files:
        raise FileNotFoundError(f"No cached BEA file found for {BEA_TABLE} in {CACHE}")
    return files[-1]


def bea_line(line_number):
    path = latest_bea_cache()
    bea = pd.read_csv(path, parse_dates=["date"])
    bea["line"] = bea["line"].astype(int)
    out = (
        bea.loc[bea["line"].eq(line_number), ["date", "desc", "value"]]
        .sort_values("date")
        .set_index("date")
    )
    if out.empty:
        raise ValueError(f"BEA line {line_number} was not found in {path.name}")
    return out["value"].astype(float).rename(out["desc"].iloc[0])


def fred_cached(series_id):
    path = CACHE / f"fred_csv_{series_id}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing cached FRED series: {path}")
    series = pickle.load(open(path, "rb")).astype(float).sort_index()
    series.index = pd.to_datetime(series.index).to_period("Q").to_timestamp(how="start")
    return series.rename(series_id)


services_index = bea_line(SERVICES_LINE).rename("services_index")
energy_index = bea_line(ENERGY_LINE).rename("energy_index")
eci_index = fred_cached(ECI_SERIES_ID).rename("eci_index")
gdpc1 = fred_cached("GDPC1").rename("gdpc1")
gdppot = fred_cached("GDPPOT").rename("gdppot")

print("BEA line 28:", services_index.name)
print("BEA line 27:", energy_index.name)
print("ECI series:", ECI_SERIES_ID)

## 2. Build the Quarterly Panel

In [ ]:
panel = pd.concat([services_index, energy_index, eci_index, gdpc1, gdppot], axis=1).sort_index()

required_sources = [services_index, energy_index, gdpc1, gdppot]
if CYCLICAL_CONTROL == "eci_wage_growth":
    required_sources.append(eci_index)

common_end = min(s.dropna().index.max() for s in required_sources)
panel = panel.loc[services_index.dropna().index.min():common_end].copy()

panel["pi"] = 100 * (panel["services_index"] / panel["services_index"].shift(1) - 1)
panel["energy_infl"] = 100 * (panel["energy_index"] / panel["energy_index"].shift(1) - 1)
panel["eci_wage_growth"] = 100 * (panel["eci_index"] / panel["eci_index"].shift(1) - 1)
panel["output_gap"] = 100 * (panel["gdpc1"] - panel["gdppot"]) / panel["gdppot"]

for lag in [1, 2]:
    panel[f"pi_l{lag}"] = panel["pi"].shift(lag)

for lag in ECI_LAGS:
    panel[f"eci_wage_growth_l{lag}"] = panel["eci_wage_growth"].shift(lag)

for lag in [1, 2, 3, 4]:
    panel[f"energy_infl_l{lag}"] = panel["energy_infl"].shift(lag)

for quarter in pd.period_range("2020Q1", "2020Q4", freq="Q"):
    panel[f"pandemic_{str(quarter).lower()}"] = (panel.index == quarter.to_timestamp()).astype(int)

panel["post_2021"] = (panel.index >= POST_START).astype(int)
panel["pi_l1_post_2021"] = panel["pi_l1"] * panel["post_2021"]
panel["pi_l2_post_2021"] = panel["pi_l2"] * panel["post_2021"]

audit_vars = [
    "services_index", "energy_index", "pi", "energy_infl",
    "output_gap", "eci_wage_growth", *ECI_TERMS,
]
sample_audit = pd.DataFrame(
    {
        "variable": col,
        "first": panel[col].dropna().index.min(),
        "last": panel[col].dropna().index.max(),
        "nonmissing": panel[col].notna().sum(),
    }
    for col in audit_vars
)

settings = pd.DataFrame(
    [
        {"setting": "cyclical_control", "value": CYCLICAL_CONTROL},
        {"setting": "control_terms", "value": " + ".join(CONTROL["terms"])},
        {"setting": "control_label", "value": CONTROL["label"]},
        {"setting": "eci_lags", "value": ", ".join(str(lag) for lag in ECI_LAGS)},
    ]
)

save_csv(panel, OUT / "quarterly_panel.csv", index_label="date")
save_csv(sample_audit, OUT / "sample_audit.csv", index=False)
save_csv(settings, OUT / "settings.csv", index=False)
display(sample_audit)

## 3. Estimate the Persistence Regressions

In [ ]:
pandemic_dummies = [f"pandemic_2020q{i}" for i in range(1, 5)]
energy_terms = [f"energy_infl_l{i}" for i in range(1, 5)]

eq1_terms = [
    "pi_l1", "pi_l2",
    *CONTROL["terms"],
    *energy_terms,
    *pandemic_dummies,
]

eq2_terms = [
    "pi_l1", "pi_l2", "pi_l1_post_2021", "pi_l2_post_2021",
    *CONTROL["terms"],
    *energy_terms,
    *pandemic_dummies,
]


def regression_frame(terms):
    return panel[["pi", *terms]].dropna()


def fit_ols(frame, terms):
    x = sm.add_constant(frame[terms], has_constant="add")
    return sm.OLS(frame["pi"], x).fit()


def qlabel(ts):
    return pd.Timestamp(ts).to_period("Q").strftime("%YQ%q")


eq1 = regression_frame(eq1_terms)
eq1_pre = eq1.loc[:PRE_PANDEMIC_END].copy()
eq1_latest = eq1.tail(len(eq1_pre)).copy()
eq2 = regression_frame(eq2_terms)

res_pre = fit_ols(eq1_pre, eq1_terms)
res_latest = fit_ols(eq1_latest, eq1_terms)
res_interaction = fit_ols(eq2, eq2_terms)

print(f"Pre-pandemic split sample: {qlabel(eq1_pre.index.min())} to {qlabel(eq1_pre.index.max())}; n={len(eq1_pre)}")
print(f"Latest equal-length sample: {qlabel(eq1_latest.index.min())} to {qlabel(eq1_latest.index.max())}; n={len(eq1_latest)}")
print(f"Interaction sample: {qlabel(eq2.index.min())} to {qlabel(eq2.index.max())}; n={len(eq2)}")

## 4. Summarize Persistence

In [ ]:
def lincom(result, terms):
    test = result.t_test(" + ".join(terms) + " = 0")
    ci = np.asarray(test.conf_int()).ravel()
    return {
        "estimate": float(np.squeeze(test.effect)),
        "se": float(np.squeeze(test.sd)),
        "t_stat": float(np.squeeze(test.tvalue)),
        "p_value": float(np.squeeze(test.pvalue)),
        "ci_low": float(ci[0]),
        "ci_high": float(ci[1]),
    }


def summary_row(model, comparison, quantity, result, frame, terms):
    row = {
        "model": model,
        "comparison": comparison,
        "quantity": quantity,
        "cyclical_control": CYCLICAL_CONTROL,
        "control_terms": " + ".join(CONTROL["terms"]),
        "terms": " + ".join(terms),
        "nobs": int(result.nobs),
        "r_squared": result.rsquared,
        "sample_start": qlabel(frame.index.min()),
        "sample_end": qlabel(frame.index.max()),
    }
    row.update(lincom(result, terms))
    return row


persistence_summary = pd.DataFrame(
    [
        summary_row(
            "eq1_pre_pandemic",
            "pre_pandemic_split_sample",
            "persistence_gamma1_plus_gamma2",
            res_pre,
            eq1_pre,
            ["pi_l1", "pi_l2"],
        ),
        summary_row(
            "eq1_latest_equal_window",
            "latest_equal_length_split_sample",
            "persistence_gamma1_plus_gamma2",
            res_latest,
            eq1_latest,
            ["pi_l1", "pi_l2"],
        ),
        summary_row(
            "eq2_post_2021_interaction",
            "formal_interaction_test",
            "pre_2021_persistence_gamma1_plus_gamma2",
            res_interaction,
            eq2,
            ["pi_l1", "pi_l2"],
        ),
        summary_row(
            "eq2_post_2021_interaction",
            "formal_interaction_test",
            "post_2021_increase_eta1_plus_eta2",
            res_interaction,
            eq2,
            ["pi_l1_post_2021", "pi_l2_post_2021"],
        ),
        summary_row(
            "eq2_post_2021_interaction",
            "formal_interaction_test",
            "post_2021_persistence_gamma_plus_eta",
            res_interaction,
            eq2,
            ["pi_l1", "pi_l2", "pi_l1_post_2021", "pi_l2_post_2021"],
        ),
    ]
)

save_csv(persistence_summary, OUT / "persistence_summary.csv", index=False)
display(persistence_summary.round(4))

In [ ]:
def coefficient_table(result, model, frame):
    ci = result.conf_int()
    return pd.DataFrame(
        {
            "model": model,
            "term": result.params.index,
            "coef": result.params.values,
            "se": result.bse.values,
            "t_stat": result.tvalues.values,
            "p_value": result.pvalues.values,
            "ci_low": ci[0].values,
            "ci_high": ci[1].values,
            "nobs": int(result.nobs),
            "r_squared": result.rsquared,
            "sample_start": qlabel(frame.index.min()),
            "sample_end": qlabel(frame.index.max()),
        }
    )


coefs = pd.concat(
    [
        coefficient_table(res_pre, "eq1_pre_pandemic", eq1_pre),
        coefficient_table(res_latest, "eq1_latest_equal_window", eq1_latest),
        coefficient_table(res_interaction, "eq2_post_2021_interaction", eq2),
    ],
    ignore_index=True,
)

save_csv(coefs, OUT / "regression_coefficients.csv", index=False)
display(coefs.query("term in ['pi_l1', 'pi_l2', 'pi_l1_post_2021', 'pi_l2_post_2021']"))

## 5. Figure 1-Style Chart

In [ ]:
figure_data = persistence_summary[persistence_summary["comparison"].str.contains("split")].copy()
figure_data["label"] = ["Pre-pandemic sample", "Sample including post-pandemic data"]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
colors = ["#2f6fb3", "#c44e52"]
x = np.array([-0.18, 0.18])

for xpos, (_, row), color in zip(x, figure_data.iterrows(), colors):
    err_low = row["estimate"] - row["ci_low"]
    err_high = row["ci_high"] - row["estimate"]
    ax.bar(
        xpos,
        row["estimate"],
        width=0.32,
        color=color,
        edgecolor="white",
        linewidth=0.8,
        yerr=[[err_low], [err_high]],
        capsize=4,
        label=row["label"],
    )
    ax.text(
        xpos,
        row["estimate"] + 0.03,
        f"{row['estimate']:.2f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.axhline(0, color="0.35", linewidth=1)
ax.set_xlim(-0.6, 0.6)
ax.set_xticks([0])
ax.set_xticklabels(["U.S. non-housing services"])
ax.set_ylabel("Sum of coefficients on lagged inflation")
ax.set_title("Inflation Persistence\n" + f"Cyclical control: {CONTROL['label']}", loc="left")
ax.legend(frameon=False, loc="upper left")

ymin = min(0, figure_data["ci_low"].min() - 0.1)
ymax = max(0.8, figure_data["ci_high"].max() + 0.12)
ax.set_ylim(ymin, ymax)

note = (
    f"Pre-pandemic: {figure_data.iloc[0]['sample_start']} to {figure_data.iloc[0]['sample_end']}. "
    f"Post-pandemic-inclusive equal window: {figure_data.iloc[1]['sample_start']} to {figure_data.iloc[1]['sample_end']}."
)
fig.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=8, color="0.35")
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIG / "figure1_style_persistence.png", dpi=180)

if matplotlib.get_backend().lower() != "agg":
    plt.show()
else:
    plt.close(fig)

## 6. Outlook Calculations

In [ ]:
FORECAST_HORIZON = 3
FUTURE_ENERGY_INFLATION = 0.0

# Scenario assumptions for future values of non-inflation controls.
# The baseline output-gap model holds the latest observed output gap constant.
# If CYCLICAL_CONTROL is "eci_wage_growth", the code holds latest observed ECI wage growth constant.
FUTURE_OUTPUT_GAP = panel["output_gap"].dropna().iloc[-1]
FUTURE_ECI_WAGE_GROWTH = panel["eci_wage_growth"].dropna().iloc[-1]


def period_map(series):
    return {
        pd.Timestamp(index).to_period("Q"): value
        for index, value in series.dropna().items()
    }


def value_for_lag(values, period, lag):
    return values.get(period - lag, np.nan)


params = res_latest.params
last_observed_period = panel["pi"].dropna().index.max().to_period("Q")
forecast_periods = [last_observed_period + step for step in range(1, FORECAST_HORIZON + 1)]

pi_values = period_map(panel["pi"])
energy_values = period_map(panel["energy_infl"])
output_gap_values = period_map(panel["output_gap"])
eci_values = period_map(panel["eci_wage_growth"])

outlook_rows = []
contribution_rows = []

for period in forecast_periods:
    term_contributions = {"constant": params.get("const", 0.0)}
    forecast = term_contributions["constant"]

    for term in eq1_terms:
        if term == "output_gap":
            value = output_gap_values.get(period, FUTURE_OUTPUT_GAP)
            bucket = "cyclical_control"
        elif term.startswith("eci_wage_growth_l"):
            lag = int(term.replace("eci_wage_growth_l", ""))
            value = value_for_lag(eci_values, period, lag)
            if pd.isna(value):
                value = FUTURE_ECI_WAGE_GROWTH
            bucket = "cyclical_control"
        elif term.startswith("energy_infl_l"):
            lag = int(term.replace("energy_infl_l", ""))
            value = value_for_lag(energy_values, period, lag)
            if pd.isna(value):
                value = FUTURE_ENERGY_INFLATION
            bucket = "energy_lags"
        elif term.startswith("pi_l"):
            lag = int(term.replace("pi_l", ""))
            value = value_for_lag(pi_values, period, lag)
            bucket = "lagged_inflation"
        elif term.startswith("pandemic_"):
            value = 0.0
            bucket = "pandemic_dummies"
        else:
            value = np.nan
            bucket = "other"

        contribution = params[term] * value
        forecast += contribution
        term_contributions[bucket] = term_contributions.get(bucket, 0.0) + contribution

        contribution_rows.append(
            {
                "quarter": str(period),
                "term": term,
                "coefficient": params[term],
                "input_value": value,
                "contribution": contribution,
                "bucket": bucket,
            }
        )

    pi_values[period] = forecast
    energy_values[period] = FUTURE_ENERGY_INFLATION
    output_gap_values[period] = FUTURE_OUTPUT_GAP
    eci_values[period] = FUTURE_ECI_WAGE_GROWTH

    outlook_rows.append(
        {
            "quarter": str(period),
            "forecast_qoq_pct": forecast,
            "annualized_x4_pct": forecast * 4,
            "constant": term_contributions.get("constant", 0.0),
            "lagged_inflation": term_contributions.get("lagged_inflation", 0.0),
            "cyclical_control": term_contributions.get("cyclical_control", 0.0),
            "energy_lags": term_contributions.get("energy_lags", 0.0),
            "pandemic_dummies": term_contributions.get("pandemic_dummies", 0.0),
        }
    )

outlook = pd.DataFrame(outlook_rows)
outlook_contributions = pd.DataFrame(contribution_rows)

save_csv(outlook, OUT / "outlook_forecast.csv", index=False)
save_csv(outlook_contributions, OUT / "outlook_contributions.csv", index=False)

display(outlook.round(3))
display(
    outlook_contributions
    .pivot_table(index="quarter", columns="bucket", values="contribution", aggfunc="sum")
    .reset_index()
    .round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
bars = ax.bar(outlook["quarter"], outlook["forecast_qoq_pct"], color="#2f6fb3", width=0.55)
ax.axhline(0, color="0.35", linewidth=1)
ax.set_title("Persistence-Implied Outlook\n" + f"Cyclical control: {CONTROL['label']}", loc="left")
ax.set_ylabel("Quarterly percent change")
ax.set_ylim(0, max(1.25, outlook["forecast_qoq_pct"].max() + 0.25))

for bar, annualized in zip(bars, outlook["annualized_x4_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f"{bar.get_height():.2f}\n({annualized:.1f}% ann.)",
        ha="center",
        va="bottom",
        fontsize=9,
    )

note = (
    "Forecast uses Equation (1), latest equal-window coefficients. "
    f"Future energy inflation assumed {FUTURE_ENERGY_INFLATION:.1f}% q/q; "
    "control held at latest observed value."
)
fig.text(0.01, 0.01, note, ha="left", va="bottom", fontsize=8, color="0.35")
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.savefig(FIG / "outlook_forecast.png", dpi=180)

if matplotlib.get_backend().lower() != "agg":
    plt.show()
else:
    plt.close(fig)

## 7. Regression Diagnostics

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.tsa.stattools import acf


def residual_diagnostics(result, model_name):
    resid = pd.Series(result.resid).dropna()
    lb_lag = min(4, max(1, len(resid) // 5))
    lb = acorr_ljungbox(resid, lags=[lb_lag], return_df=True)
    jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(resid)
    bp_stat, bp_pvalue, _, _ = het_breuschpagan(result.resid, result.model.exog)

    return {
        "model": model_name,
        "nobs": int(result.nobs),
        "r_squared": result.rsquared,
        "adj_r_squared": result.rsquared_adj,
        "rmse": float(np.sqrt(np.mean(resid**2))),
        "mae": float(np.mean(np.abs(resid))),
        "mean_residual": float(resid.mean()),
        "std_residual": float(resid.std(ddof=1)),
        "durbin_watson": float(durbin_watson(resid)),
        f"ljung_box_p_lag_{lb_lag}": float(lb["lb_pvalue"].iloc[0]),
        "jarque_bera_p": float(jb_pvalue),
        "skew": float(skew),
        "kurtosis": float(kurtosis),
        "breusch_pagan_p": float(bp_pvalue),
    }


diagnostics_summary = pd.DataFrame(
    [
        residual_diagnostics(res_latest, "eq1_latest_equal_window"),
        residual_diagnostics(res_interaction, "eq2_post_2021_interaction"),
    ]
)

save_csv(diagnostics_summary, OUT / "regression_diagnostics_summary.csv", index=False)
display(diagnostics_summary.round(4))

In [ ]:
def plot_regression_diagnostics(result, frame, title, filename, acf_lags=8):
    fitted = pd.Series(result.fittedvalues, index=frame.index)
    resid = pd.Series(result.resid, index=frame.index)
    max_lag = min(acf_lags, len(resid) - 2)
    acf_values = acf(resid, nlags=max_lag, fft=False)[1:]
    acf_index = np.arange(1, max_lag + 1)

    fig, axes = plt.subplots(2, 2, figsize=(12, 7.2))
    ax = axes[0, 0]
    ax.plot(frame.index, frame["pi"], color="#111827", linewidth=1.8, label="Actual")
    ax.plot(frame.index, fitted, color="#2f7f70", linewidth=1.8, label="Fitted")
    ax.set_title("Actual vs. Fitted Inflation")
    ax.set_ylabel("Quarterly percent change")
    ax.legend(frameon=False, loc="upper left")

    ax = axes[0, 1]
    ax.plot(frame.index, resid, color="#c15a2e", linewidth=1.6)
    ax.axhline(0, color="0.55", linewidth=1)
    ax.set_title(f"{title} Residuals")

    ax = axes[1, 0]
    ax.hist(resid, bins=12, density=True, color="#c7d1cc", edgecolor="#334155", alpha=0.9)
    sigma = resid.std(ddof=1)
    if sigma > 0:
        grid = np.linspace(resid.min(), resid.max(), 200)
        normal_pdf = np.exp(-0.5 * ((grid - resid.mean()) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
        ax.plot(grid, normal_pdf, color="#111827", linewidth=1.8)
    ax.set_title("Residual Distribution")
    ax.set_xlabel("Residual")

    ax = axes[1, 1]
    ax.bar(acf_index, acf_values, color="#5f6f91")
    ax.axhline(0, color="0.55", linewidth=1)
    ax.set_title("Residual Autocorrelation")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")
    ax.set_xticks(acf_index)

    fig.suptitle(title, x=0.01, ha="left", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(FIG / filename, dpi=180)

    if matplotlib.get_backend().lower() != "agg":
        plt.show()
    else:
        plt.close(fig)


plot_regression_diagnostics(
    res_latest,
    eq1_latest,
    "Equation (1), Latest Equal-Window Regression",
    "diagnostics_eq1_latest_equal_window.png",
)
plot_regression_diagnostics(
    res_interaction,
    eq2,
    "Equation (2), Post-2021 Interaction Regression",
    "diagnostics_eq2_post_2021_interaction.png",
)

## 8. Interpretation

In [ ]:
pre = persistence_summary.query("model == 'eq1_pre_pandemic'").iloc[0]
latest = persistence_summary.query("model == 'eq1_latest_equal_window'").iloc[0]
increase = persistence_summary.query("quantity == 'post_2021_increase_eta1_plus_eta2'").iloc[0]
post = persistence_summary.query("quantity == 'post_2021_persistence_gamma_plus_eta'").iloc[0]

print(
    f"With {CONTROL['label']} as the only cyclical control, split-sample persistence "
    f"moves from {pre.estimate:.3f} ({pre.sample_start}-{pre.sample_end}) "
    f"to {latest.estimate:.3f} ({latest.sample_start}-{latest.sample_end})."
)
print(
    f"In the interaction regression, the post-2021 increase is {increase.estimate:.3f} "
    f"(p={increase.p_value:.3f}), implying post-2021 persistence of {post.estimate:.3f}."
)
print(f"Outputs saved to: {OUT}")